In [ ]:
import os
import random
import numpy as np
import librosa
import soundfile as sf

In [ ]:
data_path = 'data_new'
output_path = 'data_extend'
noise_path = 'noisepines.wav'

sample_rate = 22050
duration = 2 

os.makedirs(output_path, exist_ok=True)

In [ ]:
def apply_mixup(audio, audio_list):
    """Mixup"""
    audio2_path = random.choice(audio_list)
    audio2, _ = librosa.load(audio2_path, sr=sample_rate, duration=duration)
    audio2 = librosa.util.fix_length(data=audio2, size=len(audio))
    lam = np.random.beta(a=0.2, b=0.2)
    return lam * audio + (1 - lam) * audio2

def get_rms(y):
    return np.sqrt(np.mean(y**2))

def apply_overlay(audio, noise_file_path):
    """
    Overlay
    """
    try:
        noise_duration_total = librosa.get_duration(path=noise_file_path)
        if duration > noise_duration_total:
            offset = 0
        else:
            offset = random.uniform(0, noise_duration_total - duration)

        noise_segment, _ = librosa.load(
            noise_file_path, sr=sample_rate, offset=offset, duration=duration
        )
        noise_segment = librosa.util.fix_length(data=noise_segment, size=len(audio))
    except Exception as e:
        print(f"process files: {noise_file_path} failed: {e}")
        return audio 

    signal_rms = get_rms(audio)
    noise_rms = get_rms(noise_segment)

    if noise_rms == 0:  
        return audio

    target_snr_db = random.uniform(-5, 5)    # -5 to 5 dB
    target_snr = 10**(target_snr_db / 10)

    scaling_factor = signal_rms / (noise_rms * np.sqrt(target_snr))

    # mix noise and audio
    return audio + noise_segment * scaling_factor


def apply_pitch_shift(audio):
    """pitch shift"""
    steps = random.uniform(-2, 2)
    return librosa.effects.pitch_shift(y=audio, sr=sample_rate, n_steps=steps)

def apply_time_shift(audio):
    """time shift"""
    max_shift_ms = 500
    max_shift_frames = int(max_shift_ms * sample_rate / 1000)
    shift = random.randint(-max_shift_frames, max_shift_frames)
    return np.roll(audio, shift)


In [ ]:
for i, audio_path in enumerate(audio_files):
    try:
        original_audio, _ = librosa.load(audio_path, sr=sample_rate, duration=duration)
        original_audio = librosa.util.fix_length(data=original_audio, size=int(sample_rate * duration))
    except Exception as e:
        print(f"load file: {audio_path} failed: {e}")
        continue 


    augmentation_methods = [
        ('mixup', lambda aud: apply_mixup(aud, audio_files)),
        ('overlay', lambda aud: apply_overlay(aud, river_noise_path)),
        ('pitch_shift', apply_pitch_shift),
        ('time_shift', apply_time_shift)
    ] 

    # randomly select one or more methods
    num_augs_to_apply = random.randint(1, len(augmentation_methods))
    selected_methods = random.sample(augmentation_methods, num_augs_to_apply)

    augmented_audio = original_audio
    method_names = []
    for method_name, func in selected_methods:
        augmented_audio = func(augmented_audio)
        method_names.append(method_name)

    original_filename = os.path.basename(audio_path)
    if '_resampled.wav' in original_filename:
        output_filename = original_filename.replace('_resampled.wav', '_generated.wav')
    else:
        base_name = original_filename.rsplit('.', 1)[0]
        output_filename = f"{base_name}_generated.wav"

    output_filepath = os.path.join(output_path, output_filename)

    # save output
    try:
        sf.write(output_filepath, augmented_audio, sample_rate)
    except Exception as e:
        print(f"save files: {output_filepath} failed: {e}")
